<a href="https://colab.research.google.com/github/rifahnanjiba02-arch/vision-language-composition/blob/main/scripts/MACCO_W1_CML_Visual_Patch_Noise_Controlled.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# MACCO W1+CML — Masking vs Gaussian Visual-Patch Noise

## Overview

This notebook starts from our already trained **W1+CML CLIP checkpoint** on Hugging Face, and adapts its visual encoder to Gaussian noise applied to the actual image pixels before CLIP normalization/encoding. The original authors' model is not retrained here.

**Training flow:** W1+CML checkpoint → reviewed COCO images/captions → clean/noisy pixel inputs → CLIP contrastive + approved-negative CML + small clean/noisy consistency objective → one CLIP-compatible `.pt` file → optional upload into `noise_adaption/input_pixel_noise/` on Hugging Face.


## 1 · Install dependencies and fetch the authors’ MACCO source

In [1]:
%pip -q install "open_clip_torch==2.32.0" "huggingface_hub>=0.25,<2.0" "datasets>=3,<5" pandas pyarrow pillow requests tqdm

In [2]:
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from contextlib import nullcontext
from io import BytesIO
from urllib.parse import quote, urlparse
import os, json, math, random, hashlib, gc

import numpy as np
import pandas as pd
import requests
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import open_clip
from huggingface_hub import hf_hub_download, HfApi

assert torch.cuda.is_available(), 'Colab: Runtime > Change runtime type > GPU'
DEVICE = torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)
print('OpenCLIP:', getattr(open_clip, '__version__', 'installed'))

GPU: Tesla T4
PyTorch: 2.11.0+cu130
OpenCLIP: 2.32.0


## 2 · Run settings
Defaults intentionally favor preserving the pretrained W1+CML model over aggressive small-data fine-tuning: freeze its text tower and most visual blocks; optimize only the last four visual transformer blocks, the final visual layer norm, and visual projection. 30% of training images remain clean, and other inputs receive progressive pixel noise.

Pixel noise standard deviations below refer to image intensities in **[0, 1]**, not to embedding-space sigma; using sigma=1.0 here would heavily damage most images.

In [3]:
#@title Training and output settings
BASE_REPO_ID = 'rifahnanjiba2/MACCO-Finetuned'
BASE_CHECKPOINT = 'MACCO-W1-CML-ViT-B-32.pt'
BASE_REVISION = 'main'

AUDIT_REPO = 'rifahnanjiba02-arch/vision-language-composition'
AUDIT_REVISION = 'main'  # Pin to a commit hash for a formally reproducible run.
AUDIT_PATH = 'results/cml_audit (1).csv'
HF_COCO_REPO = 'AbdoTW/COCO_2014'

WORK = Path('/content/macco_w1_cml_input_pixel_noise')
IMAGE_ROOT = WORK / 'images'
OUTPUT_DIR = WORK / 'outputs'
EXPORT_DIR = OUTPUT_DIR / 'hf_export'
for p in (WORK, IMAGE_ROOT, OUTPUT_DIR, EXPORT_DIR):
    p.mkdir(parents=True, exist_ok=True)

SEED = 42
REVIEWED_ROWS = 512
EPOCHS = 3
BATCH_SIZE = 8
GRAD_ACCUM = 2  # Effective batch size about 16.
NUM_WORKERS = 2
TRAIN_LAST_VISUAL_BLOCKS = 4
LR_VISUAL = 1e-6
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
MAX_BATCHES_PER_EPOCH = None  # None = all available reviewed rows.

CLEAN_FRACTION = 0.30             # Keep a clean-image subset for retention.
MIN_NONZERO_PIXEL_SIGMA = 0.02
START_MAX_PIXEL_SIGMA = 0.06
END_MAX_PIXEL_SIGMA = 0.18
CML_MARGIN = 0.05
CML_WEIGHT = 1.0
CONSISTENCY_WEIGHT = 0.15

HF_TARGET_REPO = 'rifahnanjiba2/MACCO-Finetuned'
HF_SUBDIR = 'noise_adaption/input_pixel_noise'  # Preserve your existing HF folder spelling.
HF_FILENAME = 'MACCO-W1-CML-InputPixelNoise-ViT-B-32.pt'
UPLOAD_TO_HF = False  # Set True ONLY when you want to publish the final export.

assert 0 <= CLEAN_FRACTION < 1
assert 0 < MIN_NONZERO_PIXEL_SIGMA <= START_MAX_PIXEL_SIGMA <= END_MAX_PIXEL_SIGMA <= 1
assert TRAIN_LAST_VISUAL_BLOCKS in range(1, 13)
assert EPOCHS >= 1 and BATCH_SIZE >= 2 and GRAD_ACCUM >= 1

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print('Single training run | output:', OUTPUT_DIR)

Single training run | output: /content/macco_w1_cml_input_pixel_noise/outputs


## 3 · Download the ORIGINAL W1+CML checkpoint from Hugging Face

In [4]:
from huggingface_hub import get_token
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

ckpt_path = Path(hf_hub_download(
    repo_id=BASE_REPO_ID, filename=BASE_CHECKPOINT,
    revision=BASE_REVISION, repo_type='model', token=HF_TOKEN
))
model, train_preprocess, val_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained=str(ckpt_path), device=DEVICE, precision='fp32'
)
tokenize = open_clip.tokenize
print('Starting model:', ckpt_path.name)
print('Checkpoint bytes:', ckpt_path.stat().st_size)
print('Model loading matches the OpenCLIP usage in your Table 1 notebook.')

MACCO-W1-CML-ViT-B-32.pt: reconstructing file:   0%|          |  0.00B /  605MB            

MACCO-W1-CML-ViT-B-32.pt: downloading bytes:           |  0.00B            

Starting model: MACCO-W1-CML-ViT-B-32.pt
Checkpoint bytes: 605222779
Model loading matches the OpenCLIP usage in your Table 1 notebook.


## 4 · Reviewed image–caption rows and CML negatives
Keep only the same **512** reviewed COCO rows from your earlier experiment and use a negative caption only when `approved=True`. The audit file contains `filepath`, `title`, `candidate_negative`, and `approved (along with `row_id`). Rows without approved negatives are still useful for contrastive training.

In [5]:
audit_url = (f'https://raw.githubusercontent.com/{AUDIT_REPO}/'
             f'{quote(AUDIT_REVISION, safe="")}/'
             f'{quote(AUDIT_PATH, safe="/")}')
audit_resp = requests.get(audit_url, timeout=90)
audit_resp.raise_for_status()
AUDIT_LOCAL = WORK / 'reviewed_cml_audit.csv'
AUDIT_LOCAL.write_bytes(audit_resp.content)
audit = pd.read_csv(AUDIT_LOCAL, keep_default_na=False)
needed = {'row_id', 'filepath', 'title', 'candidate_negative', 'approved'}
assert needed.issubset(audit.columns), f'Missing audit columns: {needed - set(audit.columns)}'
audit['approved'] = audit['approved'].astype(str).str.strip().str.lower().isin(
    ('true', '1', 'yes', 'y'))
assert not audit[['filepath','title']].duplicated().any(), 'Audit contains repeated image/caption rows.'
assert (audit['filepath'].astype(str).str.startswith('coco/train2014/')).all(), (
    'Unexpected COCO split in audit; inspect before training.')
assert not (audit.approved & audit.candidate_negative.astype(str).str.strip().eq('')).any()
assert not (audit.approved & (audit.candidate_negative.str.strip() == audit.title.str.strip())).any()

train_rows = audit.sort_values('row_id').head(REVIEWED_ROWS).copy().reset_index(drop=True)
assert len(train_rows) == REVIEWED_ROWS, 'Not enough reviewed rows.'
train_rows['positive_caption'] = train_rows.title.astype(str)
train_rows['negative_caption'] = np.where(
    train_rows.approved, train_rows.candidate_negative.astype(str), train_rows.title.astype(str))
print('Reviewed rows:', len(train_rows))
print('Approved negatives:', int(train_rows.approved.sum()))
print('Distinct image paths:', train_rows.filepath.nunique())
display(train_rows[['filepath','title','candidate_negative','approved']].head(3))

Reviewed rows: 512
Approved negatives: 107
Distinct image paths: 512


,filepath,title,candidate_negative,approved
0,coco/train2014/COCO_train2014_000000052826.jpg,some street signs sitting by a wall,,False
1,coco/train2014/COCO_train2014_000000401846.jpg,Two surfers at the beach walking into the waves.,,False
2,coco/train2014/COCO_train2014_000000073215.jpg,Antique cars on the grass in front of an antiq...,Antique cars on the grass behind an antique tr...,True



## 5 · Download only images needed for training
Try the Hugging Face COCO-2014 mirror with **filtered streaming**, then retry any missing original `train2014` filenames from the official COCO image server. Cache verified images so a Colab rerun does not redownload everything.

If an image still cannot be recovered, that row is excluded and recorded explicitly in the data manifest. The notebook will not silently pretend all 512 images were retrieved.


In [6]:
from datasets import load_dataset, get_dataset_split_names

wanted = defaultdict(list)
all_relative_paths = sorted(train_rows.filepath.astype(str).unique())

def verified_image(path):
    if not path.is_file():
        return False
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        path.unlink(missing_ok=True)
        return False

for rel in all_relative_paths:
    path = IMAGE_ROOT / rel
    if not verified_image(path):
        coco_id = int(Path(rel).stem.rsplit('_', 1)[-1])
        wanted[coco_id].append(rel)

print('Already cached:', len(all_relative_paths) - sum(map(len, wanted.values())))
print('Still missing:', sum(map(len, wanted.values())))

if wanted:
    try:
        splits = get_dataset_split_names(HF_COCO_REPO)
        for split in ('train', 'validation', 'test'):
            if not wanted: break
            if split not in splits: continue
            print('Trying HF split:', split)
            ds = load_dataset(
                HF_COCO_REPO, split=split, streaming=True,
                filters=[('id', 'in', sorted(wanted))], token=HF_TOKEN
            )
            for row in tqdm(ds, desc=f'HF COCO {split}'):
                iid = int(row['id'])
                if iid not in wanted:
                    continue
                filename = Path(urlparse(str(row.get('coco_url') or '')).path).name
                matching = [rel for rel in wanted[iid] if Path(rel).name == filename]
                if not matching:
                    continue
                pic = row['image']
                if isinstance(pic, dict):
                    pic = Image.open(BytesIO(pic['bytes'])) if pic.get('bytes') else Image.open(pic['path'])
                with pic:
                    rgb = pic.convert('RGB')
                    for rel in matching:
                        dst = IMAGE_ROOT / rel
                        dst.parent.mkdir(parents=True, exist_ok=True)
                        rgb.save(dst, format='JPEG', quality=95)
                        assert verified_image(dst)
                for rel in matching:
                    wanted[iid].remove(rel)
                if not wanted[iid]:
                    del wanted[iid]
                if not wanted:
                    break
    except Exception as error:
        print('HF mirror incomplete or unavailable:', repr(error))
        print('Using direct original COCO file fallback for remaining images.')

still_missing = [rel for rels in wanted.values() for rel in rels]
print('Images requiring original COCO fallback:', len(still_missing))

def official_coco_download(rel):
    name = Path(rel).name
    url = f'https://images.cocodataset.org/train2014/{name}'
    path = IMAGE_ROOT / rel
    try:
        response = requests.get(url, timeout=(10, 35))
        response.raise_for_status()
        with Image.open(BytesIO(response.content)) as check:
            check.verify()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(response.content)
        return rel, verified_image(path), ''
    except Exception as error:
        return rel, False, str(error)[:160]

if still_missing:
    with ThreadPoolExecutor(max_workers=6) as pool:
        futures = [pool.submit(official_coco_download, rel) for rel in still_missing]
        failures = []
        for future in tqdm(as_completed(futures), total=len(futures), desc='COCO fallback'):
            rel, ok, err = future.result()
            if not ok:
                failures.append((rel, err))
else:
    failures = []

available = {rel for rel in all_relative_paths if verified_image(IMAGE_ROOT / rel)}
train_rows['image_available'] = train_rows.filepath.isin(available)
manifest_path = OUTPUT_DIR / 'training_image_manifest.csv'
train_rows.to_csv(manifest_path, index=False)
missing_details = OUTPUT_DIR / 'missing_training_images.json'
missing_details.write_text(json.dumps(failures, indent=2), encoding='utf-8')
train_rows = train_rows[train_rows.image_available].reset_index(drop=True)

print('Training rows with images:', len(train_rows), '/', REVIEWED_ROWS)
print('Approved CML negatives retained:', int(train_rows.approved.sum()))
print('Manifest:', manifest_path)
assert len(train_rows) >= int(REVIEWED_ROWS * 0.80), (
    'Fewer than 80% of reviewed rows have images; investigate download source before training.')
assert train_rows.approved.sum() >= 10, 'Too few approved negatives remain.'

Already cached: 0
Still missing: 512


README.md:   0%|          | 0.00/808 [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Trying HF split: train


Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

HF COCO train: 0it [00:00, ?it/s]

Images requiring original COCO fallback: 0
Training rows with images: 512 / 512
Approved CML negatives retained: 107
Manifest: /content/macco_w1_cml_input_pixel_noise/outputs/training_image_manifest.csv


## 6 · Dataloader (no masks or decoders)
We use the deterministic CLIP center-crop preprocessing instead of random training crops, reducing the risk of removing the captioned object in this small compositionality dataset. For repeated COCO image paths, a batch may contain more than one caption for the same image. Training uses a **multi-positive contrastive** loss rather than incorrectly treating another caption of the same image as a negative.

In [7]:
class ReviewedCOCODataset(Dataset):
    def __init__(self, frame, image_root, transform):
        self.frame = frame.reset_index(drop=True)
        self.root = image_root
        self.transform = transform
        self.pos_tokens = tokenize(self.frame.positive_caption.tolist())
        self.neg_tokens = tokenize(self.frame.negative_caption.tolist())
        paths = self.frame.filepath.astype(str).tolist()
        image_id_map = {path: idx for idx, path in enumerate(sorted(set(paths)))}
        self.image_ids = [image_id_map[p] for p in paths]

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        record = self.frame.iloc[idx]
        with Image.open(self.root / str(record.filepath)) as pic:
            clean_x = self.transform(pic.convert('RGB'))
        return (
            clean_x, self.pos_tokens[idx], self.neg_tokens[idx],
            bool(record.approved), self.image_ids[idx]
        )

train_dataset = ReviewedCOCODataset(train_rows, IMAGE_ROOT, val_preprocess)
loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=bool(NUM_WORKERS), generator=loader_generator,
    drop_last=False
)
print('Images/captions:', len(train_dataset), '| batches per epoch:', len(train_loader))
x, pos, neg, approved, ids = next(iter(train_loader))
print('Input shape:', tuple(x.shape), '| token shape:', tuple(pos.shape))
print('Approved negatives in first batch:', int(approved.sum()))

Images/captions: 512 | batches per epoch: 64
Input shape: (8, 3, 224, 224) | token shape: (8, 77)
Approved negatives in first batch: 1


## 7 · Gaussian noise directly on input pixels
The OpenCLIP preprocessing pipeline normalizes the image. We first undo that normalization`[0,1]` pixel scale, clamp, and normalize back. The entire input image can be noisy (not just annotations-selected patches).

No image-patch masking, text masking, reconstruction decoder, or original MACCO source patching is involved.

In [8]:
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(1, 3, 1, 1)

def add_gaussian_pixel_noise(normalized_images, sigma_per_image):
    """Add noise to raw [0,1] pixels, then return CLIP-normalized tensors."""
    mean = CLIP_MEAN.to(device=normalized_images.device, dtype=normalized_images.dtype)
    std = CLIP_STD.to(device=normalized_images.device, dtype=normalized_images.dtype)
    sigma = sigma_per_image.to(device=normalized_images.device,
                               dtype=normalized_images.dtype).view(-1, 1, 1, 1)
    assert sigma.shape[0] == normalized_images.shape[0]
    pixels = (normalized_images * std + mean).clamp(0.0, 1.0)
    noisy = (pixels + torch.randn_like(pixels) * sigma).clamp(0.0, 1.0)
    return (noisy - mean) / std

# Verify that sigma=0 changes only negligible boundary rounding, and sigma>0 changes pixels.
test_x = x[:2].to(DEVICE)
unchanged = add_gaussian_pixel_noise(test_x, torch.zeros(2, device=DEVICE))
perturbed = add_gaussian_pixel_noise(test_x, torch.tensor([0.10, 0.10], device=DEVICE))
print('No-noise maximum pixel difference:', float((unchanged-test_x).abs().max()))
print('Noisy mean pixel difference:', float((perturbed-test_x).abs().mean()))
assert torch.isfinite(perturbed).all()
assert (perturbed-test_x).abs().mean() > 0

No-noise maximum pixel difference: 1.1920928955078125e-07
Noisy mean pixel difference: 0.28879162669181824


## 8 · Freeze text and most visual weights; set up the three non-masking losses
We optimize:
- **Image–text contrastive alignment**, with all same-image captions counted as positives;
- **CML approved-negative hinge loss**, comparing each image's true caption with its reviewed incorrect caption;
- **Small clean/noisy feature consistency**, so a noisy view stays close to its clean view (the clean reference is detached).

These are intentionally not the original W1 masked-reconstruction losses. The point is to preserve useful W1+CML checkpoint knowledge while learning noise robustness without training the author's masking procedure again.

In [9]:
# Freeze whole model first; keep the base text encoder and logit scale intact.
for param in model.parameters():
    param.requires_grad_(False)

visual_blocks = model.visual.transformer.resblocks
assert len(visual_blocks) == 12, 'Expected ViT-B/32 with 12 transformer blocks.'
for block in visual_blocks[-TRAIN_LAST_VISUAL_BLOCKS:]:
    for param in block.parameters():
        param.requires_grad_(True)
for param in model.visual.ln_post.parameters():
    param.requires_grad_(True)
if isinstance(model.visual.proj, torch.nn.Parameter):
    model.visual.proj.requires_grad_(True)
elif isinstance(model.visual.proj, torch.nn.Module):
    for param in model.visual.proj.parameters():
        param.requires_grad_(True)

trainable = [p for p in model.parameters() if p.requires_grad]
assert trainable, 'No parameters selected for training.'
print('Trainable parameters:', sum(p.numel() for p in trainable))
print('Frozen text tower: yes | training last visual blocks:', TRAIN_LAST_VISUAL_BLOCKS)

def multi_positive_contrastive(image_features, text_features, image_ids, scale):
    # Multiple captions of the same COCO image should not compete as negatives.
    sim = scale * (image_features @ text_features.T)
    positive = image_ids[:, None].eq(image_ids[None, :])
    neg_inf = torch.finfo(sim.dtype).min
    image_loss = (torch.logsumexp(sim, dim=1) -
                  torch.logsumexp(sim.masked_fill(~positive, neg_inf), dim=1)).mean()
    text_loss = (torch.logsumexp(sim, dim=0) -
                 torch.logsumexp(sim.masked_fill(~positive, neg_inf), dim=0)).mean()
    return 0.5 * (image_loss + text_loss)

def calculate_losses(images, clean_images, pos_tokens, neg_tokens, approved, image_ids):
    # Only image encoder gradients; text features fixed to original W1+CML checkpoint.
    with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
        pos_features = F.normalize(model.encode_text(pos_tokens).float(), dim=-1)
        neg_features = (F.normalize(model.encode_text(neg_tokens[approved]).float(), dim=-1)
                        if approved.any() else None)
    with torch.autocast('cuda', dtype=torch.float16):
        image_features = F.normalize(model.encode_image(images).float(), dim=-1)
        scale = model.logit_scale.exp().detach().clamp(max=100)
        contrast = multi_positive_contrastive(image_features, pos_features, image_ids, scale)
        if approved.any():
            pos_cos = (image_features[approved] * pos_features[approved]).sum(-1)
            neg_cos = (image_features[approved] * neg_features).sum(-1)
            cml = F.relu(CML_MARGIN + neg_cos - pos_cos).mean()
        else:
            cml = image_features.sum() * 0.0

        # Clean target is detached; this encourages noisy-view stability.
        noisy_mask = (images - clean_images).abs().flatten(1).mean(1) > 1e-7
        if CONSISTENCY_WEIGHT > 0 and noisy_mask.any():
            with torch.no_grad():
                clean_features = F.normalize(
                    model.encode_image(clean_images[noisy_mask]).float(), dim=-1)
            consistency = (1.0 - (image_features[noisy_mask] * clean_features).sum(-1)).mean()
        else:
            consistency = image_features.sum() * 0.0
        total = contrast + CML_WEIGHT*cml + CONSISTENCY_WEIGHT*consistency
    return dict(total=total, contrast=contrast, cml=cml, consistency=consistency)

# Quick gradient-safe smoke test: does not update the model.
model.train()
smoke_x = x[:min(4,len(x))].to(DEVICE)
smoke_p = pos[:len(smoke_x)].to(DEVICE)
smoke_n = neg[:len(smoke_x)].to(DEVICE)
smoke_a = approved[:len(smoke_x)].to(DEVICE)
smoke_i = ids[:len(smoke_x)].to(DEVICE)
smoke_sigmas = torch.full((len(smoke_x),), 0.07, device=DEVICE)
smoke_input = add_gaussian_pixel_noise(smoke_x, smoke_sigmas)
smoke_losses = calculate_losses(smoke_input, smoke_x, smoke_p, smoke_n, smoke_a, smoke_i)
assert all(torch.isfinite(v).item() for v in smoke_losses.values())
smoke_losses['total'].backward()
assert any(p.grad is not None for p in trainable), 'Noise training does not reach visual weights.'
model.zero_grad(set_to_none=True)
print('Smoke test losses:', {k: round(float(v.detach()), 4) for k,v in smoke_losses.items()})

Trainable parameters: 28746240
Frozen text tower: yes | training last visual blocks: 4
Smoke test losses: {'total': 0.0074, 'contrast': 0.0001, 'cml': 0.0, 'consistency': 0.0491}


## 9 · One run: W1+CML → pixel-noise fine-tuning
There is no author-model arm, no mask-control model, no masked warm-up and no evaluation here. For each batch, randomly retain 30% of images clean; use a mix of noise magnitudes for the rest. Noise becomes stronger gradually across training. Gradient accumulation and mixed precision make the run more suitable for Colab GPUs.

The training data and actual per-batch noise range are logged. Only the final CLIP-only checkpoint is exported (no giant per-epoch checkpoints).

In [10]:
#@title Run a single input-pixel-noise adaptation training
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

steps_per_epoch = min(len(train_loader), MAX_BATCHES_PER_EPOCH or len(train_loader))
assert steps_per_epoch >= 1
num_batches = EPOCHS * steps_per_epoch
total_updates = EPOCHS * math.ceil(steps_per_epoch / GRAD_ACCUM)
warmup_updates = min(5, max(1, total_updates // 10))

optimizer = AdamW(trainable, lr=LR_VISUAL, weight_decay=WEIGHT_DECAY)

def lr_factor(index):
    update = index + 1
    if update <= warmup_updates:
        return update / warmup_updates
    fraction = (update - warmup_updates) / max(1, total_updates - warmup_updates)
    return 0.1 + 0.9 * (1.0 + math.cos(math.pi * min(1.0, fraction))) / 2.0

scheduler = LambdaLR(optimizer, lr_factor)
scaler = torch.amp.GradScaler('cuda')
optimizer.zero_grad(set_to_none=True)
step_records = []
epoch_records = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_metrics = []
    for batch_idx, batch in enumerate(tqdm(train_loader, total=steps_per_epoch,
                                           desc=f'Pixel-noise epoch {epoch}/{EPOCHS}')):
        if batch_idx >= steps_per_epoch:
            break
        clean, positives, negatives, approved_flags, image_ids = (
            v.to(DEVICE, non_blocking=True) for v in batch
        )
        global_batch = (epoch - 1) * steps_per_epoch + batch_idx
        progress = global_batch / max(1, num_batches - 1)
        max_sigma = START_MAX_PIXEL_SIGMA + progress * (END_MAX_PIXEL_SIGMA - START_MAX_PIXEL_SIGMA)
        sigmas = MIN_NONZERO_PIXEL_SIGMA + (
            max_sigma - MIN_NONZERO_PIXEL_SIGMA
        ) * torch.rand(len(clean), device=DEVICE)
        clean_mask = torch.rand(len(clean), device=DEVICE) < CLEAN_FRACTION
        sigmas[clean_mask] = 0.0
        input_images = add_gaussian_pixel_noise(clean, sigmas)

        losses = calculate_losses(input_images, clean, positives, negatives,
                                  approved_flags, image_ids)
        assert torch.isfinite(losses['total']), f'Nonfinite loss on batch {global_batch}'

        window_start = (batch_idx // GRAD_ACCUM) * GRAD_ACCUM
        window_size = min(GRAD_ACCUM, steps_per_epoch - window_start)
        scaler.scale(losses['total'] / window_size).backward()
        if (batch_idx+1) % GRAD_ACCUM == 0 or batch_idx+1 == steps_per_epoch:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable, GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        row = {
            'epoch': epoch, 'global_batch': global_batch + 1,
            'batch_size': len(clean),
            'approved_negatives_in_batch': int(approved_flags.sum()),
            'clean_fraction_actual': float(clean_mask.float().mean()),
            'pixel_sigma_mean': float(sigmas.mean()),
            'pixel_sigma_max': float(sigmas.max()),
            'noise_schedule_max': float(max_sigma),
            'total_loss': float(losses['total'].detach()),
            'contrastive_loss': float(losses['contrast'].detach()),
            'cml_loss': float(losses['cml'].detach()),
            'consistency_loss': float(losses['consistency'].detach()),
            'lr': optimizer.param_groups[0]['lr'],
        }
        step_records.append(row)
        epoch_metrics.append(row)

    epoch_df = pd.DataFrame(epoch_metrics)
    epoch_record = {
        'epoch': epoch, 'batches': len(epoch_metrics),
        'training_rows': len(train_dataset),
        **{k: float(epoch_df[k].mean()) for k in (
            'total_loss', 'contrastive_loss', 'cml_loss',
            'consistency_loss', 'pixel_sigma_mean', 'clean_fraction_actual')}
    }
    epoch_records.append(epoch_record)
    pd.DataFrame(step_records).to_csv(OUTPUT_DIR / 'training_batches.csv', index=False)
    pd.DataFrame(epoch_records).to_csv(OUTPUT_DIR / 'training_epochs.csv', index=False)
    display(pd.DataFrame([epoch_record]).round(4))

print('Training complete. Table 1 evaluation has intentionally NOT been run.')

Pixel-noise epoch 1/3:   0%|          | 0/64 [00:00<?, ?it/s]

/tmp/ipykernel_730/1260757724.py:59: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


,epoch,batches,training_rows,total_loss,contrastive_loss,cml_loss,consistency_loss,pixel_sigma_mean,clean_fraction_actual
0,1,64,512,0.1039,0.0641,0.0353,0.0301,0.0337,0.3203


Pixel-noise epoch 2/3:   0%|          | 0/64 [00:00<?, ?it/s]

,epoch,batches,training_rows,total_loss,contrastive_loss,cml_loss,consistency_loss,pixel_sigma_mean,clean_fraction_actual
0,2,64,512,0.0843,0.0467,0.031,0.0446,0.0496,0.293


Pixel-noise epoch 3/3:   0%|          | 0/64 [00:00<?, ?it/s]

,epoch,batches,training_rows,total_loss,contrastive_loss,cml_loss,consistency_loss,pixel_sigma_mean,clean_fraction_actual
0,3,64,512,0.0908,0.0473,0.0341,0.062,0.0649,0.3027


Training complete. Table 1 evaluation has intentionally NOT been run.


## 10 · Export one CLIP-compatible model + reproducibility files
The `.pt` export contains only the OpenCLIP model's `state_dict()`, not a full MACCO trainer, optimizer, decoder, or per-epoch checkpoints. We also save the configuration and training log. No accuracy/robustness metric is claimed until the separate evaluation notebook is run.

In [11]:
import shutil

export_path = EXPORT_DIR / HF_FILENAME
state_on_cpu = {key: value.detach().cpu() for key, value in model.state_dict().items()}
torch.save(state_on_cpu, export_path)
del state_on_cpu

base_file_sha = hashlib.sha256()
with ckpt_path.open('rb') as file:
    for chunk in iter(lambda: file.read(8*1024*1024), b''):
        base_file_sha.update(chunk)

run_config = {
    'trained_at_utc': datetime.now(timezone.utc).isoformat(),
    'architecture': 'open_clip ViT-B-32',
    'starting_checkpoint_repo': BASE_REPO_ID,
    'starting_checkpoint_file': BASE_CHECKPOINT,
    'starting_checkpoint_revision': BASE_REVISION,
    'starting_checkpoint_sha256': base_file_sha.hexdigest(),
    'source_audit_url': audit_url,
    'source_audit_sha256': hashlib.sha256(AUDIT_LOCAL.read_bytes()).hexdigest(),
    'image_dataset': HF_COCO_REPO,
    'reviewed_rows_requested': REVIEWED_ROWS,
    'reviewed_rows_trained': int(len(train_rows)),
    'approved_negatives_trained': int(train_rows.approved.sum()),
    'seed': SEED,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'gradient_accumulation': GRAD_ACCUM,
    'trained_visual_blocks': TRAIN_LAST_VISUAL_BLOCKS,
    'text_encoder_frozen': True,
    'learning_rate': LR_VISUAL,
    'weight_decay': WEIGHT_DECAY,
    'clean_fraction': CLEAN_FRACTION,
    'pixel_sigma_nonzero_min': MIN_NONZERO_PIXEL_SIGMA,
    'pixel_sigma_schedule_max_start': START_MAX_PIXEL_SIGMA,
    'pixel_sigma_schedule_max_end': END_MAX_PIXEL_SIGMA,
    'contrastive_loss': 'multi_positive_bidirectional_image_text',
    'cml_margin': CML_MARGIN,
    'cml_weight': CML_WEIGHT,
    'consistency_weight': CONSISTENCY_WEIGHT,
    'masked_pretraining_or_masking_control': False,
    'table1_evaluated': False,
}
(EXPORT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
shutil.copy2(OUTPUT_DIR / 'training_epochs.csv', EXPORT_DIR / 'training_epochs.csv')
shutil.copy2(OUTPUT_DIR / 'training_batches.csv', EXPORT_DIR / 'training_batches.csv')
shutil.copy2(OUTPUT_DIR / 'training_image_manifest.csv', EXPORT_DIR / 'training_image_manifest.csv')
(EXPORT_DIR / 'README.md').write_text(f"""# MACCO W1+CML — Input Pixel Noise Adaptation

This checkpoint starts from `{BASE_REPO_ID}/{BASE_CHECKPOINT}` and fine-tunes its
visual CLIP encoder on input pixel Gaussian noise. It is **not** the original
MACCO checkpoint and does **not** re-run MACCO masking/reconstruction training.

- Architecture: OpenCLIP ViT-B/32.
- Data: {len(train_rows)} reviewed COCO2014 train image–caption rows with
  {int(train_rows.approved.sum())} approved CML negatives.
- Objective: multi-positive image–caption contrastive loss + reviewed-caption
  CML hinge + a small clean/noisy consistency term.
- Frozen: CLIP text tower; early visual transformer blocks.
- Trained: last {TRAIN_LAST_VISUAL_BLOCKS} visual transformer blocks, visual norm/projection.
- Noise: Gaussian pixel noise on [0,1] images, not patch-embedding noise.
- Results: not evaluated on Table 1 yet. No improvement claim.

See `run_config.json`, `training_epochs.csv`, and
`training_image_manifest.csv` for exact run details.
""", encoding='utf-8')

print('Saved:', export_path)
print('Size (MiB):', round(export_path.stat().st_size / 1024**2, 1))
print('HF export folder:', EXPORT_DIR)
print('Files:', [p.name for p in sorted(EXPORT_DIR.iterdir())])

Saved: /content/macco_w1_cml_input_pixel_noise/outputs/hf_export/MACCO-W1-CML-InputPixelNoise-ViT-B-32.pt
Size (MiB): 577.2
HF export folder: /content/macco_w1_cml_input_pixel_noise/outputs/hf_export
Files: ['MACCO-W1-CML-InputPixelNoise-ViT-B-32.pt', 'README.md', 'run_config.json', 'training_batches.csv', 'training_epochs.csv', 'training_image_manifest.csv']


## 11· Verify export before uploading
Perform a fresh OpenCLIP load from the newly saved `.pt`. This prevents uploading a file that only the custom training model could read.

In [12]:
# This is CPU-based so the trained model can stay on GPU.
probe_model, _, _ = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained=str(export_path), device='cpu', precision='fp32'
)
assert probe_model.visual.conv1.weight.shape == model.visual.conv1.weight.shape
print('PASS: Export loads as standard OpenCLIP ViT-B/32.')
del probe_model
gc.collect()

PASS: Export loads as standard OpenCLIP ViT-B/32.


219

In [13]:
@torch.no_grad()
def evaluate_common_gaussian(m,arm,when):
    # Every arm is tested on exactly the same Gaussian image-patch noise levels.
    m.eval()
    records=[]
    for sigma in EVAL_SIGMAS:
        seed_all(SEED+911+int(sigma*1000))
        m.CLIP.visual.image_corruption_mode='noise'
        m.CLIP.visual.image_noise_strength=sigma
        all_img=[]; all_txt=[]; mse_list=[]
        for x,t,negative,tf,of,im in eval_batches:
            x,t,im=x.to(DEVICE),t.to(DEVICE),im.to(DEVICE)
            no_text_corruption=torch.zeros_like(t,dtype=torch.bool)
            with torch.autocast('cuda',dtype=torch.float16):
                clean_img,_,corrupt_img,_=m.CLIP.encode_image(x,im,m.img_mask_token)
                txt,*_=m.CLIP.encode_text(t,no_text_corruption,m.text_mask_token)
            all_img.append(F.normalize(corrupt_img.float(),dim=-1).cpu())
            all_txt.append(F.normalize(txt.float(),dim=-1).cpu())
        im_feats=torch.cat(all_img)
        txt_feats=torch.cat(all_txt)
        sim=im_feats@txt_feats.T
        diag=sim.diag()
        rank=(sim.argmax(dim=1)==torch.arange(sim.shape[0])).float().mean().item()
        records.append({'arm':arm,'stage':when,'sigma':sigma,
                        'heldout_images':len(eval_items),'image_to_text_R_at_1':rank,
                        'paired_cosine':float(diag.mean().item())})
    m.CLIP.visual.image_corruption_mode='mask'
    m.CLIP.visual.image_noise_strength=0.0
    return records

In [14]:
#@title Additional held-out image-patch reconstruction test (small, fixed subset)
RECON_ROWS = 16    # Keep this small; reconstruction heads use more GPU memory than retrieval.

@torch.no_grad()
def evaluate_image_reconstruction(m,arm,when):
    m.eval()
    results=[]
    for sigma in EVAL_SIGMAS:
        seed_all(SEED+1211+int(sigma*1000))
        m.CLIP.visual.image_corruption_mode='noise'
        m.CLIP.visual.image_noise_strength=float(sigma)
        losses=[];valid=0
        # 16 cached held-out examples, in batches no larger than training BATCH_SIZE.
        eval_subset=eval_items[:RECON_ROWS]
        for off in range(0,len(eval_subset),BATCH_SIZE):
            chunk=eval_subset[off:off+BATCH_SIZE]
            xs,ts,tflags,imflags=(torch.stack([row[j] for row in chunk]).to(DEVICE)
                                    for j in (0,1,3,5))
            with torch.autocast('cuda',dtype=torch.float16):
                output=m(xs,ts,imflags,tflags)
                # Only selected image patch targets count, same as original MACCO.
                patch_mse=criterion._compute_mask_loss_image(
                    output['pred_image'],output['target_image'],output['image_mask_index'],32)
            selected_images=int((imflags[:,1:].sum(dim=1)>0).sum())
            if selected_images:
                losses.append((float(patch_mse.item()),selected_images))
                valid+=selected_images
        mse=(sum(val*n for val,n in losses)/valid) if valid else float('nan')
        results.append({'arm':arm,'stage':when,'sigma':sigma,
                        'heldout_images':len(eval_subset),'images_with_selected_patches':valid,
                        'selected_patch_reconstruction_MSE':mse})
    m.CLIP.visual.image_corruption_mode='mask'
    m.CLIP.visual.image_noise_strength=0.0
    return results

## 12 · Optional Hugging Face upload (single final model)
Run this only when you are satisfied with the completed training.

In [16]:
#@title Upload trained model to Hugging Face

from google.colab import userdata
from huggingface_hub import HfApi

HF_TOKEN = userdata.get("HF_TOKEN")

api = HfApi(token=HF_TOKEN)

api.upload_file(
    path_or_fileobj=str(export_path),
    path_in_repo=f"{HF_SUBDIR}/{HF_FILENAME}",
    repo_id="rifahnanjiba2/MACCO-Finetuned",
    repo_type="model",
    commit_message="Upload W1+CML input-pixel-noise adapted model"
)

print("Model uploaded successfully!")
print(
    f"https://huggingface.co/rifahnanjiba2/MACCO-Finetuned/blob/main/"
    f"{HF_SUBDIR}/{HF_FILENAME}"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...putPixelNoise-ViT-B-32.pt:   3%|2         | 16.0MB /  605MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


Model uploaded successfully!
https://huggingface.co/rifahnanjiba2/MACCO-Finetuned/blob/main/noise_adaption/input_pixel_noise/MACCO-W1-CML-InputPixelNoise-ViT-B-32.pt


## After training
The only checkpoint you need for Table 1 is:

`noise_adaption/input_pixel_noise/MACCO-W1-CML-InputPixelNoise-ViT-B-32.pt`

Later, use your existing `MACCO_Table1_Evaluation.ipynb` and replace its W1+CML checkpoint filename with the remote path above. Run the authors' MACCO checkpoint and this newly adapted checkpoint on the same benchmarks. Table 1 evaluation, results CSV merging, and any claim about improvement are deliberately excluded from this training notebook.